# 11 — High-gamma probe: gamma_ij = 12, 16, 20, 24 timing calibration

Real timing data beyond the validated `gamma_ij <= 8` range, to calibrate the
very rough runtime extrapolation used to plan gamma_ij = 32/48/64 scans (see
`AI_HANDOFF.md`'s "Locked GPU Filon convergence parameters" and "Implemented:
minimal-memory GPU rewrite"). We have never run `bubblemaster_gpu` above
gamma_ij=8 -- this is the first real test beyond that range.

Unlike an earlier draft of this notebook, `t_max` here is **domain-shared**,
not per-pair: we fix a realization scale `gamma_star_max = 9.6` (so
`gamma_ij,max = 2.5 * 9.6 = 24`, matching the agreed `gamma_ij <= 2.5*gamma_*`
production convention) and compute `T_MAX = 1.4 * R_*(gamma_star_max)` **once**,
shared across all four `gamma_ij` setups -- exactly how `09_adaptive_bm_scan.ipynb`
builds a real production scan's `times[]` arrays, and different from treating
each `gamma_ij` as its own domain (which the first draft of this notebook did).

Runs **4 setups on 4 GPUs of one node**: `gamma_ij in {12, 16, 20, 24}`, all at
`dt = 2` (the actual target cutoff-time spacing for production scans, not the
coarser `dt=10/5` used in the first draft).

Note the shared `T_MAX` means `n_t` *shrinks* with increasing `gamma_ij`
(higher pair-gamma collides later within the same fixed realization budget,
leaving less time to sample before `T_MAX`) -- verified below before writing
any setups, since gamma_ij=24 is close enough to `T_MAX` that this needs
checking, not assuming.

**Workflow**
1. Run **Section 1** to compute kinematics (`t_first`, shared `T_MAX`, `n_t`)
   for each `gamma_ij`, and confirm all four are valid (`t_first < T_MAX`).
2. Run **Section 2** to write the 4 setup files.
3. Run **Section 3** to generate the SLURM script (1 node, 4 tasks, 1 GPU
   each). Submit it and wait for it to finish.
4. Run **Section 4** to parse wall-clock timing per run, for comparison
   against the gamma_ij <= 8 extrapolation.

In [1]:
from pathlib import Path
from types import SimpleNamespace

import h5py
import numpy as np
from scipy.optimize import brentq

import ptbuilder as pt
from ptbuilder.ic import BubbleMasterParams, write_2d_setup
from ptbuilder.weight_scan_diagnostics import load_kinematics

REPO_ROOT   = Path(pt.__file__).parent.parent
SCRIPTS_DIR = REPO_ROOT / 'scripts'
LOGS_DIR    = REPO_ROOT / 'logs'
SCRIPTS_DIR.mkdir(exist_ok=True)
LOGS_DIR.mkdir(exist_ok=True)

LAMBDA_BAR = 0.84

# Domain (realization) target -- NOT per-pair. gamma_ij,max = PAIR_GAMMA_FACTOR
# * GAMMA_STAR_MAX is the agreed production relation (AI_HANDOFF.md).
GAMMA_STAR_MIN    = 1.0
GAMMA_STAR_MAX    = 9.6
PAIR_GAMMA_FACTOR = 2.5

GAMMA_IJ_TARGETS = (12, 16, 20, 24)
DT = 2   # actual target cutoff-time spacing (not the coarser dt=10/5 probe)

KR_MIN, KR_MAX = 1.0, 100.0   # matches AI_HANDOFF's agreed frequency interval
N_OMEGA = 32
N_K     = 51

TIME_FACTOR = 1.4   # agreed production cutoff: T_MAX = 1.4 * R_*(gamma_star_max)

# Locked GPU parameters (AI_HANDOFF.md) -- unchanged, reused as-is.
GPU_FILON_N_MIN          = 128
GPU_FILON_PANELS_PER_OSC = 40
GPU_S_BATCH_SIZE         = 128

N_WALL_POINTS = 20    # matches the validated wp20/gs2-64 setups
HOW_OFTEN_DS  = 5
BABY_STEPS    = 100
CUTOFF_TYPE   = 0

GPU_ACCOUNT  = 'm3166_g'
GPU_WALLTIME = '04:00:00'   # generous -- gamma_ij>8 is untested territory

TAG = f'lb{LAMBDA_BAR:g}_high_gamma_probe'
scan_root  = REPO_ROOT / 'data' / TAG
setup_dir  = scan_root / 'setups'
output_dir_root = scan_root / 'gpu'
setup_dir.mkdir(parents=True, exist_ok=True)
output_dir_root.mkdir(parents=True, exist_ok=True)

instanton_path = REPO_ROOT / f'data/phi4_lambda_bar{LAMBDA_BAR:g}/instanton.h5'
kin = load_kinematics(instanton_path)


class CachedPhi4Potential:
    def __init__(self, lambda_bar):
        self.params = {'lambda_bar': float(lambda_bar)}

    def to_hdf5(self, group):
        group.attrs.create('type', 'phi4', dtype=h5py.string_dtype(encoding='ascii'))
        group.attrs['phi_false'] = 0.0
        group.attrs['phi_true']  = 1.0
        group.attrs['lambda_bar'] = self.params['lambda_bar']


model = SimpleNamespace(instanton=kin.profile, kinematics=kin,
                        potential=CachedPhi4Potential(LAMBDA_BAR))


def time_at_gamma(gamma):
    if gamma <= 1.0:
        return 0.0
    return brentq(lambda t: float(kin.Gamma(t)) - gamma, 0.0, 1.0e7)


def rstar_at_gamma_star(gamma_star):
    t = time_at_gamma(gamma_star)
    return 2.0 * float(kin.R(t, r='mid'))


pair_gamma_max = PAIR_GAMMA_FACTOR * GAMMA_STAR_MAX
if max(GAMMA_IJ_TARGETS) > pair_gamma_max:
    raise ValueError(
        f'requested gamma_ij up to {max(GAMMA_IJ_TARGETS)} exceeds '
        f'gamma_ij,max = {PAIR_GAMMA_FACTOR}*{GAMMA_STAR_MAX} = {pair_gamma_max} '
        'implied by GAMMA_STAR_MAX -- raise GAMMA_STAR_MAX or lower the targets'
    )

RSTAR_MAX = rstar_at_gamma_star(GAMMA_STAR_MAX)
T_MAX     = TIME_FACTOR * RSTAR_MAX          # shared across all four setups
OMEGA_MIN = KR_MIN / RSTAR_MAX

print(f'gamma_star range: {GAMMA_STAR_MIN} .. {GAMMA_STAR_MAX}  '
      f'-> gamma_ij,max = {pair_gamma_max}')
print(f'R_*(gamma_star_max) = {RSTAR_MAX:.3f}')
print(f'Shared T_MAX = {TIME_FACTOR}*R_* = {T_MAX:.3f}')
print(f'Global omega_min = {OMEGA_MIN:.6g}')

gamma_star range: 1.0 .. 9.6  -> gamma_ij,max = 24.0
R_*(gamma_star_max) = 133.145
Shared T_MAX = 1.4*R_* = 186.402
Global omega_min = 0.00751063


## 1. Kinematics: `t_first`/`n_t` for each `gamma_ij` (shared `T_MAX`)

Validates every target before writing anything: each pair's own collision
time (`t_first`) must be strictly less than the shared `T_MAX`, or that
`gamma_ij` cannot be scanned within this domain at all (matches the same
check `09_adaptive_bm_scan.ipynb` makes per-slice).

In [2]:
print(f"{'gamma_ij':>9}  {'t_first':>9}  {'T_MAX (shared)':>14}  {'valid':>6}  {'n_t':>4}  {'omega_max':>10}")
print('-' * 60)

combos = []   # gamma_ij, t_first, n_t, times, omega
for g in GAMMA_IJ_TARGETS:
    t_first = time_at_gamma(g)
    if t_first >= T_MAX:
        raise RuntimeError(
            f'gamma_ij={g}: t_first={t_first:.2f} >= shared T_MAX={T_MAX:.2f} '
            '-- collision happens after the domain cutoff; lower gamma_ij or '
            'raise GAMMA_STAR_MAX')
    n_t   = int(np.ceil((T_MAX - t_first) / DT)) + 1
    times = np.linspace(t_first, T_MAX, n_t)

    # Same non-rectangular frequency-domain rule as the real production scan:
    # a pair with this gamma_ij can only occur in a realization with
    # gamma_star >= max(GAMMA_STAR_MIN, gamma_ij/PAIR_GAMMA_FACTOR), which sets
    # its own omega_max via that realization's (smaller) R_*.
    smallest_target = max(GAMMA_STAR_MIN, g / PAIR_GAMMA_FACTOR)
    omega_max = KR_MAX / rstar_at_gamma_star(smallest_target)
    omega = np.geomspace(OMEGA_MIN, omega_max, N_OMEGA)

    combos.append(dict(gamma_ij=g, t_first=t_first, n_t=n_t, times=times, omega=omega))
    print(f"{g:>9}  {t_first:>9.2f}  {T_MAX:>14.2f}  {'yes':>6}  {n_t:>4}  {omega_max:>10.4g}")

 gamma_ij    t_first  T_MAX (shared)   valid   n_t   omega_max
------------------------------------------------------------
       12      82.93          186.40     yes    53       1.502
       16     110.74          186.40     yes    39       1.127
       20     138.53          186.40     yes    25      0.9013
       24     166.30          186.40     yes    12      0.7511


## 2. Write setups

In [3]:
params = BubbleMasterParams(
    dz=None, wall_points=N_WALL_POINTS,
    n_w=N_OMEGA, n_k=N_K, how_often_ds=HOW_OFTEN_DS,
    baby_steps=BABY_STEPS, cutoff_type=CUTOFF_TYPE, t_0_scal=1.0,
    collision_radius='mid',
)

manifest_rows = []
for combo in combos:
    g = combo['gamma_ij']
    name  = f'g{g:04d}_dt{DT}'
    setup = setup_dir / f'{name}.h5'
    write_2d_setup(model, g, combo['times'], setup, params, wlist=combo['omega'])
    with h5py.File(setup, 'a') as f:
        f.attrs['dt_requested']        = DT
        f.attrs['gamma_star_min']      = GAMMA_STAR_MIN
        f.attrs['gamma_star_max']      = GAMMA_STAR_MAX
        f.attrs['pair_gamma_factor']   = PAIR_GAMMA_FACTOR
        f.attrs['probe_note'] = (
            'High-gamma timing probe -- T_MAX shared across the domain '
            '(gamma_star_max), not per-pair; see gamma_star_min/max attrs.'
        )
    manifest_rows.append((g, DT, setup.relative_to(REPO_ROOT), name))
    print(f'Written: {setup.relative_to(REPO_ROOT)}  (n_t={combo["n_t"]})')

manifest = scan_root / 'manifest.tsv'
manifest.write_text(
    'gamma_ij\tdt\tsetup\tname\n' +
    ''.join(f'{g}\t{dt}\t{s}\t{n}\n' for g, dt, s, n in manifest_rows)
)
print(f'\nManifest: {manifest.relative_to(REPO_ROOT)}')

Written: data/lb0.84_high_gamma_probe/setups/g0012_dt2.h5  (n_t=53)
Written: data/lb0.84_high_gamma_probe/setups/g0016_dt2.h5  (n_t=39)
Written: data/lb0.84_high_gamma_probe/setups/g0020_dt2.h5  (n_t=25)
Written: data/lb0.84_high_gamma_probe/setups/g0024_dt2.h5  (n_t=12)

Manifest: data/lb0.84_high_gamma_probe/manifest.tsv


## 3. Generate SLURM script -- 1 node, 4 tasks, 1 GPU each

Four independent runs, one per GPU, launched concurrently in the background
and waited on.

In [4]:
manifest_rel = manifest.relative_to(REPO_ROOT)
script_path  = SCRIPTS_DIR / f'{TAG}_gpu.sh'

script_path.write_text(f'''#!/bin/bash
#SBATCH --job-name=bm_{TAG}
#SBATCH --output={LOGS_DIR.relative_to(REPO_ROOT)}/bm_{TAG}_%j.out
#SBATCH --constraint=gpu
#SBATCH --qos=regular
#SBATCH -A {GPU_ACCOUNT}
#SBATCH --nodes=1
#SBATCH --ntasks=4
#SBATCH --ntasks-per-node=4
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-task=1
#SBATCH --time={GPU_WALLTIME}
set -euo pipefail
REPO_ROOT="${{SLURM_SUBMIT_DIR:-$PWD}}"
cd "$REPO_ROOT"
pids=()
while IFS=$'\\t' read -r gamma dt setup name; do
    [[ "$gamma" == "gamma_ij" ]] && continue
    output="{output_dir_root.relative_to(REPO_ROOT)}/${{name}}"
    mkdir -p "$output"
    /usr/bin/time --verbose srun --exclusive --exact -N 1 -n 1 -c 32 --gpus-per-task=1 --cpu-bind=cores bin/bubblemaster_gpu "$setup" "$output/" --param {GPU_FILON_N_MIN} --filon-panels-per-osc {GPU_FILON_PANELS_PER_OSC} --s-batch-size {GPU_S_BATCH_SIZE} &
    pids+=("$!")
done < {manifest_rel}
status=0
for pid in "${{pids[@]}}"; do
    wait "$pid" || status=1
done
exit "$status"
''')
script_path.chmod(0o755)
print(f'Written: {script_path.relative_to(REPO_ROOT)}')
print(f'Submit with:  sbatch {script_path.relative_to(REPO_ROOT)}')

Written: scripts/lb0.84_high_gamma_probe_gpu.sh
Submit with:  sbatch scripts/lb0.84_high_gamma_probe_gpu.sh


## 4. Parse wall-clock timing

Matches each `/usr/bin/time --verbose` block to its `gamma_ij` by the
output-directory string in the command line, not by order -- the 4 tasks run
concurrently and share one SLURM log, so line order is not reliable.

In [5]:
import re


def parse_walltime_seconds(text_after_command):
    m = re.search(r'Elapsed \(wall clock\) time.*?:\s+(\S+)', text_after_command)
    if not m:
        return None
    parts = [float(x) for x in m.group(1).split(':')]
    if len(parts) == 2:
        return parts[0]*60 + parts[1]
    return parts[0]*3600 + parts[1]*60 + parts[2]


def parse_timed_blocks(log_path):
    text = Path(log_path).read_text(errors='replace')
    blocks = []
    for chunk in text.split('\tCommand being timed: ')[1:]:
        cmd_line, _, rest = chunk.partition('\n')
        blocks.append((cmd_line.strip().strip('\"'), parse_walltime_seconds(rest)))
    return blocks


def latest_log(pattern):
    logs = sorted(LOGS_DIR.glob(pattern), key=lambda p: p.stat().st_mtime)
    return logs[-1] if logs else None


timing_by_name = {}
log = latest_log(f'bm_{TAG}_*.out')
if log is not None:
    for cmd, secs in parse_timed_blocks(log):
        m = re.search(rf'{re.escape(str(output_dir_root.relative_to(REPO_ROOT)))}/([^/\"]+)/', cmd)
        if m and secs is not None:
            timing_by_name[m.group(1)] = secs
    print(f'Parsed {len(timing_by_name)}/{len(combos)} timings from {log.name}')
else:
    print(f'No log found yet matching bm_{TAG}_*.out')

print()
print(f"{'gamma_ij':>9}  {'n_t':>4}  {'wall time':>12}")
print('-' * 30)
for combo in combos:
    name = f"g{combo['gamma_ij']:04d}_dt{DT}"
    secs = timing_by_name.get(name)
    secs_str = f'{secs:.1f}s' if secs is not None else 'NOT READY'
    print(f"{combo['gamma_ij']:>9}  {combo['n_t']:>4}  {secs_str:>12}")

No log found yet matching bm_lb0.84_high_gamma_probe_*.out

 gamma_ij   n_t     wall time
------------------------------
       12    53     NOT READY
       16    39     NOT READY
       20    25     NOT READY
       24    12     NOT READY
